In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, theta_0, x_r, theta_r):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))
    d["theta_r"].append(theta_r.round(4))

In [ ]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset):
    alpha = params['alpha']
    lamb = params['lamb']
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    
    n = len(X)
    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        x_r = recourse.get_recourse(x_0)
        append_result(results, recourse.name, seed, alpha, lamb, i, x_0, x_r, theta_0)

    df_results = pd.DataFrame(results)
    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f'../results/recourse/lr_{dataset.name}_{recourse.name}_{seed}.pkl')
    
    return df_results

In [ ]:
lr_lambda = lambda last_epoch: (last_epoch+1) **(-0.5)
optimizer = torch.optim.SGD([torch.tensor([1.,], requires_grad=True)], lr=2.5)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
prev_lr = 2.5
for epoch in range(100):
    optimizer.step()
    scheduler.step()
    if scheduler.get_lr()[0] <= 1e-3:
        print(epoch)
        break
    print(scheduler.get_lr()[0])
    prev_lr = scheduler.get_lr()[0]

1.7677669529663689
1.4433756729740643
1.25
1.118033988749895
1.0206207261596576
0.9449111825230682
0.8838834764831844
0.8333333333333333
0.7905694150420949
0.7537783614444091
0.7216878364870322
0.6933752452815365
0.6681531047810609
0.6454972243679027
0.625
0.6063390625908325
0.5892556509887896
0.5735393346764044
0.5590169943749475
0.545544725589981
0.5330017908890261
0.5212860351426869
0.5103103630798288
0.5
0.4902903378454601
0.4811252243246882
0.4724555912615341
0.46423834544262965
0.4564354645876384
0.4490132550669373
0.4419417382415922
0.4351941398892446
0.4287464628562721
0.42257712736425823
0.41666666666666663
0.41099746826339323
0.40555355282690636
0.40032038451271784
0.39528470752104744
0.3904344047215152
0.3857583749052298
0.3812464258315117
0.37688918072220456
0.37267799624996495
0.36860489038724287
0.3646624787447364
0.3608439182435161
0.3571428571428571
0.35355339059327373
0.3500700210070024
0.34668762264076824
0.3434014098717226
0.34020690871988585
0.3370999312316211
0.334

In [5]:
def run_experiment(dataset: Dataset, recourse_fn: Recourse, params: dict, results: List):
    alpha = params['alpha']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha)
        if params["lamb"] is None:
            params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
            recourse.lamb = params['lamb']
        
        df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
        results.append(df_results)

In [6]:
torch.manual_seed(0)

d_results = {}
params = {}
params['alpha'] = 0.5 # float, None
params['lamb'] = 0.1
params['seeds'] = range(5)
params['save_results'] = False

datasets = [SyntheticDataset(), SBADataset()]
recourse_fns = [ROAR]

for dataset in datasets:
    results = []
    print(f'Running {dataset.name} data...')
    for recourse_fn in recourse_fns:
        run_experiment(dataset, recourse_fn, params, results)
    
    d_results[dataset.name] = pd.concat(results)
    print(f'Finished {dataset.name}\n')

Running synthetic data...


Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 96/96 [00:55<00:00,  1.74it/s]
Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 95/95 [00:54<00:00,  1.73it/s]
Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 103/103 [00:59<00:00,  1.73it/s]
Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 101/101 [00:58<00:00,  1.72it/s]
Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 105/105 [01:00<00:00,  1.73it/s]


Finished synthetic

Running sba data...


Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 39/39 [00:58<00:00,  1.50s/it]
Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 36/36 [00:56<00:00,  1.56s/it]
Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 40/40 [01:00<00:00,  1.51s/it]
Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 36/36 [00:53<00:00,  1.50s/it]
Evaluating recourse | alpha=0.5; lambda=0.1: 100%|██████████| 38/38 [01:01<00:00,  1.62s/it]

Finished sba

